# Handling Missing Values Using Drop and Fill Strategies

## Learning Objectives
- Understand different strategies for handling missing data
- Drop missing values safely when appropriate
- Fill missing values using suitable methods
- Understand trade-offs between dropping and filling
- Make intentional decisions based on data context

---

## By Completing This Milestone, You Will Be Able To:
- Remove rows or columns with missing data when necessary
- Fill missing values using constants or simple statistics
- Choose the right strategy based on the situation
- Avoid introducing bias or errors unintentionally
- Prepare clean data for analysis or modeling

---

## Why This Matters

**Handling missing data incorrectly can be worse than leaving it untouched.**

Common beginner issues:
- Dropping too much data without realizing the impact
- Filling missing values blindly
- Mixing strategies without understanding consequences
- Distorting analysis results due to poor handling choices

**This milestone ensures that:**
- Missing data is handled intentionally
- Data integrity is preserved as much as possible
- Cleaning steps are justified and explainable
- Analysis results are more trustworthy

Think of missing data handling as making **informed trade-offs**, not quick fixes.

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np

print("✓ Libraries imported successfully!")

---

## Part 1: Load and Identify Missing Values

### Loading Data with Missing Values

Let's load the employee survey dataset that contains real missing values and understand what we're working with.

In [ ]:
# Load the employee survey data with missing values
df = pd.read_csv('../data/raw/employee_survey_with_missing_2026_Q1.csv')

print("✓ Data loaded successfully!")
print(f"\nDataset shape: {df.shape}")
print(f"Total rows: {df.shape[0]}")
print(f"Total columns: {df.shape[1]}")
print(f"\nColumn names:")
print(df.columns.tolist())

### Identifying Missing Values

Before handling missing data, we must understand where and how many missing values exist.

In [ ]:
# Check for missing values using isnull()
print("Missing values by column:")
print("=" * 60)
missing_counts = df.isnull().sum()
missing_percent = (df.isnull().sum() / len(df)) * 100

for col in df.columns:
    if missing_counts[col] > 0:
        print(f"{col:20s} | Missing: {missing_counts[col]:2d} ({missing_percent[col]:5.1f}%)")

print("\n" + "=" * 60)
print(f"Total missing values in dataset: {df.isnull().sum().sum()}")
print(f"Columns with missing data: {(missing_counts > 0).sum()}")
print(f"Rows with at least one missing value: {df.isnull().any(axis=1).sum()}")

In [ ]:
# Use info() to see missing values in context of data types
print("\nDataFrame info:")
print("=" * 60)
df.info()

# Display first few rows to see missing values
print("\n" + "=" * 60)
print("First 10 rows (notice missing values):")
print("=" * 60)
print(df.head(10))

---

## Part 2: Dropping Missing Values - Strategy 1

### When to Drop Rows with Missing Values

**Drop rows when:**
- The row has critical missing values
- Missing data is completely random
- You can afford to lose information
- The percentage is small (< 5%)

**Do NOT drop rows when:**
- Missing data is systematic (pattern exists)
- Each row contains valuable information
- Missing percentage is high (> 20%)

### Drop Rows: Using dropna()

`dropna()` removes rows (or columns) containing ANY missing values.

In [ ]:
# Strategy 1: Drop ALL rows with ANY missing values
df_dropped_strict = df.dropna()

print("STRATEGY 1: Drop rows with ANY missing values")
print("=" * 60)
print(f"Original shape:     {df.shape}")
print(f"Shape after dropna: {df_dropped_strict.shape}")
print(f"Rows removed:       {df.shape[0] - df_dropped_strict.shape[0]}")
print(f"Data retained:      {(df_dropped_strict.shape[0] / df.shape[0] * 100):.1f}%")
print(f"\nWarning: We lost {100 - (df_dropped_strict.shape[0] / df.shape[0] * 100):.1f}% of data!")
print(f"This is TOO aggressive for this dataset.")

In [ ]:
# Strategy 2: Drop rows only if missing values in specific columns (more selective)
df_dropped_selective = df.dropna(subset=['employee_id', 'department', 'survey_date', 'satisfaction_score'])

print("\n" + "=" * 60)
print("STRATEGY 2: Drop only if CRITICAL columns are missing")
print("=" * 60)
print("Critical columns: employee_id, department, survey_date, satisfaction_score")
print(f"Original shape:        {df.shape}")
print(f"Shape after selective: {df_dropped_selective.shape}")
print(f"Rows removed:          {df.shape[0] - df_dropped_selective.shape[0]}")
print(f"Data retained:         {(df_dropped_selective.shape[0] / df.shape[0] * 100):.1f}%")
print(f"\nThis approach is MORE reasonable - we keep {(df_dropped_selective.shape[0] / df.shape[0] * 100):.1f}% of data")

---

## Part 3: Dropping Columns with Missing Values - Strategy 2

### When to Drop Columns with Missing Values

**Drop columns when:**
- Column has > 50% missing values
- Column is redundant
- Column is not needed for analysis

**Do NOT drop columns when:**
- Column is critical to analysis
- Missing percentage is mostly low
- Information can be recovered through imputation

### Drop Columns with Excessive Missing Data

Use `dropna(axis=1)` to remove columns with missing values.

In [ ]:
# Drop columns that have ANY missing values
df_cols_dropped = df.dropna(axis=1)

print("STRATEGY 3: Drop columns with ANY missing values")
print("=" * 60)
print(f"Original columns: {df.shape[1]}")
print(f"After dropping:   {df_cols_dropped.shape[1]}")
print(f"Columns removed:  {df.shape[1] - df_cols_dropped.shape[1]}")
print(f"Remaining columns: {df_cols_dropped.columns.tolist()}")
print(f"\nWarning: This removes ALL non-critical columns!")
print(f"Risk: We lose valuable data about work_life_balance, team_collaboration, etc.")

In [ ]:
# More selective: Drop only columns with > 20% missing values
threshold = 0.2 * len(df)  # 20% threshold
cols_to_drop = [col for col in df.columns if df[col].isnull().sum() > threshold]

df_cols_selective = df.drop(columns=cols_to_drop)

print("\n" + "=" * 60)
print("STRATEGY 4: Drop columns with > 20% missing values (BETTER)")
print("=" * 60)
print(f"Columns with > 20% missing: {cols_to_drop}")
print(f"Original columns: {df.shape[1]}")
print(f"After dropping:   {df_cols_selective.shape[1]}")
print(f"Columns removed:  {len(cols_to_drop)}")
print(f"\nRemaining columns: {df_cols_selective.columns.tolist()}")
print(f"This is a better approach - keep columns with usable data!")

---

## Part 4: Filling Missing Values with Constants - Strategy 3

### When to Use Constant Fill

**Use constants when:**
- Missing value has clear meaning (e.g., 0 for "no", "Unknown" for category)
- You want explicit representation of missing data
- Domain knowledge tells you the right value
- Very few missing values

**Examples:**
- Department: Fill with "Unknown"
- Boolean field: Fill with False (0)
- Optional response: Fill with "Not provided"

### Fill with Constants

`fillna(value)` replaces NaN values with specified value.

In [ ]:
# Strategy 5: Fill missing categorical values with "Unknown"
df_const_fill = df.copy()
df_const_fill['department'] = df_const_fill['department'].fillna('Unknown')

print("STRATEGY 5: Fill missing categorical values with constant")
print("=" * 60)
print(f"Original missing values in department: {df['department'].isnull().sum()}")
print(f"After fillna('Unknown'):              {df_const_fill['department'].isnull().sum()}")
print(f"\nDepartment value counts:")
print(df_const_fill['department'].value_counts())
print(f"\nNote: We now have a clear 'Unknown' category representing missing departments")

In [ ]:
# Fill multiple columns with different constants using fillna()
df_multi_fill = df.copy()
df_multi_fill.fillna({
    'department': 'Unknown',
    'survey_date': '2026-02-15',  # Mid-point of survey period
    'satisfaction_score': 5,      # Middle of 1-10 scale
    'response_text': 'No response'
}, inplace=True)

print("\n" + "=" * 60)
print("STRATEGY 6: Fill multiple columns with domain-specific constants")
print("=" * 60)
print("Before filling:")
print(f"  department NaN:        {df['department'].isnull().sum()}")
print(f"  survey_date NaN:       {df['survey_date'].isnull().sum()}")
print(f"  satisfaction_score NaN: {df['satisfaction_score'].isnull().sum()}")
print(f"  response_text NaN:     {df['response_text'].isnull().sum()}")

print("\nAfter filling with constants:")
print(f"  department NaN:        {df_multi_fill['department'].isnull().sum()}")
print(f"  survey_date NaN:       {df_multi_fill['survey_date'].isnull().sum()}")
print(f"  satisfaction_score NaN: {df_multi_fill['satisfaction_score'].isnull().sum()}")
print(f"  response_text NaN:     {df_multi_fill['response_text'].isnull().sum()}")

print("\nTotal missing values after filling:", df_multi_fill.isnull().sum().sum())

---

## Part 5: Filling Missing Values with Statistics - Strategy 4

### When to Use Statistical Fill

**Use statistics when:**
- Numeric data with missing values
- Missing values are likely random
- Distribution preservation is important
- Many missing values (> 5%)

**Options:**
- **Mean**: Average value, sensitive to outliers
- **Median**: Middle value, robust to outliers (PREFERRED)
- **Mode**: Most frequent value, works for categorical
- **Forward fill**: Last known value (for time series)
- **Backward fill**: Next known value (for time series)

### Fill with Mean or Median

Use aggregation functions to compute statistics then fill.

In [ ]:
# Examine satisfaction_score distribution
print("STRATEGY 7: Fill numeric columns with statistical measures")
print("=" * 60)
print("\nAnalyzing 'satisfaction_score' column:")
print(f"Mean:   {df['satisfaction_score'].mean():.2f}")
print(f"Median: {df['satisfaction_score'].median():.2f}")
print(f"Mode:   {df['satisfaction_score'].mode().values[0]:.0f}")
print(f"Min:    {df['satisfaction_score'].min():.0f}")
print(f"Max:    {df['satisfaction_score'].max():.0f}")
print(f"\nMissing values: {df['satisfaction_score'].isnull().sum()}")
print(f"Original distribution:\n{df['satisfaction_score'].value_counts().sort_index()}")

In [ ]:
# Fill missing values with different statistics
df_mean_fill = df.copy()
df_mean_fill['satisfaction_score'].fillna(df['satisfaction_score'].mean(), inplace=True)

df_median_fill = df.copy()
df_median_fill['satisfaction_score'].fillna(df['satisfaction_score'].median(), inplace=True)

df_mode_fill = df.copy()
df_mode_fill['satisfaction_score'].fillna(df['satisfaction_score'].mode()[0], inplace=True)

print("\n" + "=" * 60)
print("Comparing fill strategies for satisfaction_score:")
print("=" * 60)

print(f"\n1. Fill with MEAN ({df['satisfaction_score'].mean():.2f}):")
print(f"   Missing values after: {df_mean_fill['satisfaction_score'].isnull().sum()}")
print(f"   Distribution:\n{df_mean_fill['satisfaction_score'].value_counts().sort_index()}")

print(f"\n2. Fill with MEDIAN ({df['satisfaction_score'].median():.2f}):")
print(f"   Missing values after: {df_median_fill['satisfaction_score'].isnull().sum()}")
print(f"   Distribution:\n{df_median_fill['satisfaction_score'].value_counts().sort_index()}")

print(f"\n3. Fill with MODE ({df['satisfaction_score'].mode()[0]:.0f}):")
print(f"   Missing values after: {df_mode_fill['satisfaction_score'].isnull().sum()}")
print(f"   Distribution:\n{df_mode_fill['satisfaction_score'].value_counts().sort_index()}")

print("\n✓ MEDIAN is generally preferred - it's robust to outliers!")

In [ ]:
# Fill using forward fill and backward fill (useful for time series)
df_ffill = df.copy()
df_ffill['department'].fillna(method='ffill', inplace=True)

df_bfill = df.copy()
df_bfill['department'].fillna(method='bfill', inplace=True)

print("\n" + "=" * 60)
print("Time-Series Fill Methods (for ordered data):")
print("=" * 60)

print("\nOriginal department column (first 8 rows):")
print(df['department'].head(8).to_string())

print("\nAfter FORWARD FILL (ffill):")
print(df_ffill['department'].head(8).to_string())

print("\nAfter BACKWARD FILL (bfill):")
print(df_bfill['department'].head(8).to_string())

print("\nNote: Forward/backward fill works best for time-series or sequential data.")
print("Use with caution - makes assumptions about missing values!")

---

## Part 6: Comparing Drop vs Fill Strategies

### Strategy Comparison Matrix

Let's evaluate different strategies on our dataset:

In [ ]:
import pandas as pd

# Create comparison summary
strategies = {
    'Original': df,
    'Drop All (axis=0)': df_dropped_strict,
    'Drop Critical Cols': df_dropped_selective,
    'Fill Constants': df_const_fill,
    'Fill w/ Median': df_median_fill,
}

print("COMPREHENSIVE STRATEGY COMPARISON")
print("=" * 80)
print(f"{'Strategy':<25} {'Rows':<8} {'Cols':<8} {'Missing':<12} {'Data%':<10} {'Pros/Cons':<25}")
print("=" * 80)

for name, data in strategies.items():
    rows = data.shape[0]
    cols = data.shape[1]
    missing = data.isnull().sum().sum()
    data_pct = (rows / df.shape[0]) * 100
    
    if name == 'Original':
        status = "Baseline"
    elif 'Drop All' in name:
        status = "✗ Too aggressive"
    elif 'Drop Critical' in name:
        status = "✓ Better balance"
    elif 'Constants' in name:
        status = "✓ Explicit"
    elif 'Median' in name:
        status = "✓✓ Best for numeric"
    
    print(f"{name:<25} {rows:<8} {cols:<8} {missing:<12} {data_pct:<10.1f} {status:<25}")

print("\n" + "=" * 80)

In [ ]:
# Decision guide: What to use when
print("\nDECISION GUIDE: WHEN TO USE EACH STRATEGY")
print("=" * 80)

decision_guide = """
┌─────────────────────────────────────────────────────────────────────────────┐
│ STEP 1: Assess the amount of missing data                                   │
├─────────────────────────────────────────────────────────────────────────────┤
│  < 1% missing    → DROP (information loss is minimal)                        │
│  1-5% missing    → FILL (with median for numeric, mode for categorical)      │
│  5-20% missing   → FILL (with statistics) or DROP (if data is critical)      │
│  > 20% missing   → CONSIDER DROPPING COLUMN entirely                         │
├─────────────────────────────────────────────────────────────────────────────┤
│ STEP 2: Consider data type and domain                                        │
├─────────────────────────────────────────────────────────────────────────────┤
│  NUMERIC data         → Fill with MEDIAN (robust to outliers)                │
│  CATEGORICAL data     → Fill with MODE or meaningful category ("Unknown")    │
│  TEXT/RESPONSE data   → Keep missing or fill with "No response"              │
│  CRITICAL columns     → Fill rather than drop (lose business logic)           │
│  OPTIONAL columns     → Drop if too many missing values                       │
└─────────────────────────────────────────────────────────────────────────────┘
"""
print(decision_guide)

---

## Part 7: Avoiding Common Mistakes

### Common Pitfalls

1. **MISTAKE: Dropping too aggressively without understanding impact**
   - ❌ Bad: `df.dropna()` without checking how much data you lose
   - ✓ Good: Check impact first, drop only non-critical rows with few missing values

2. **MISTAKE: Filling categorical data with numeric values**
   - ❌ Bad: `df['department'].fillna(0)` - now you have 0 as a department!
   - ✓ Good: `df['department'].fillna('Unknown')` - explicitly mark missing

3. **MISTAKE: Using mean for skewed distributions**
   - ❌ Bad: Mean is pulled by outliers
   - ✓ Good: Use median which is more robust

4. **MISTAKE: Hiding missing data issues**
   - ❌ Bad: Fill data then forget about it
   - ✓ Good: Document what you filled and why

5. **MISTAKE: Not checking results after cleaning**
   - ❌ Bad: Run fill() and assume it worked
   - ✓ Good: Always verify results

### Validation Best Practices

In [ ]:
print("\nVALIDATION CHECKLIST: After Handling Missing Values")
print("=" * 80)

# Create a recommended cleaned dataset
df_recommended = df.copy()

# Strategy: Drop only critical rows, fill numeric with median, categorical with mode
df_recommended = df_recommended.dropna(subset=['employee_id', 'department', 'satisfaction_score'])

# Fill remaining missing values strategically
df_recommended['survey_date'].fillna(df_recommended['survey_date'].mode()[0], inplace=True)
df_recommended['work_life_balance'].fillna(df_recommended['work_life_balance'].median(), inplace=True)
df_recommended['management_support'].fillna(df_recommended['management_support'].median(), inplace=True)
df_recommended['career_growth'].fillna(df_recommended['career_growth'].median(), inplace=True)
df_recommended['team_collaboration'].fillna(df_recommended['team_collaboration'].median(), inplace=True)
df_recommended['response_text'].fillna('No response provided', inplace=True)

print("\n✓ VALIDATION STEP 1: Check no missing values remain")
print(f"Total missing values: {df_recommended.isnull().sum().sum()}")
print(f"Status: {'✓ PASS - No missing values' if df_recommended.isnull().sum().sum() == 0 else '✗ FAIL - Still has missing values'}")

print("\n✓ VALIDATION STEP 2: Verify shape and size")
print(f"Original shape:      {df.shape}")
print(f"Cleaned shape:       {df_recommended.shape}")
print(f"Rows retained:       {df_recommended.shape[0]}/{df.shape[0]} ({(df_recommended.shape[0]/df.shape[0]*100):.1f}%)")
print(f"Status: {'✓ PASS - Acceptable data loss' if (df_recommended.shape[0]/df.shape[0]) > 0.8 else '✗ WARNING - Lost too much data'}")

print("\n✓ VALIDATION STEP 3: Check data types unchanged")
print("Data types before vs after:")
print(df[['satisfaction_score', 'work_life_balance']].dtypes)
print(df_recommended[['satisfaction_score', 'work_life_balance']].dtypes)

print("\n✓ VALIDATION STEP 4: Spot check some rows")
print("\nSample of cleaned data (first 5 rows):")
print(df_recommended.head())

---

## Part 8: Summary and Key Takeaways

### What We Learned

**Handling missing values is a critical data cleaning task that requires intentional decisions.**

#### Strategy Summary

| Strategy | Best For | Pros | Cons |
|----------|----------|------|------|
| **Drop rows** | Few missing values (< 1%) | Simple, preserves data integrity | Loses information |
| **Drop columns** | > 50% missing in column | Simple, removes unreliable data | Loses entire features |
| **Fill with constant** | Categorical data | Explicit, interpretable | Makes assumptions |
| **Fill with median** | Numeric data | Robust to outliers, preserves distribution | Makes assumptions |
| **Fill with mode** | Categorical data | Meaningful for categories | Increases mode frequency |
| **Forward/Backward fill** | Time series data | Maintains temporal patterns | Only for ordered data |

#### Best Practices

1. **Always inspect missing data first** - Understand the pattern
2. **Choose strategy based on amount** - 1% vs 50% requires different approaches
3. **Consider data type** - Numeric vs categorical need different methods
4. **Document your choices** - Explain why you dropped or filled each column
5. **Validate results** - Always check that cleaning worked as expected
6. **Preserve data integrity** - Avoid introducing bias through poor choices

#### The Golden Rule

**No single strategy is "best" for all situations. Good data cleaning is thoughtful, explainable, and based on your specific data context.**

---

### Recommended Approach for This Dataset

For the employee survey data:
- ✓ Drop rows missing critical fields (employee_id, department, satisfaction_score)
- ✓ Fill numeric scores with **median** (robust to outliers)
- ✓ Fill categorical fields with mode or "Unknown"
- ✓ Fill open text responses with "No response provided"
- ✓ Keep 88% of data while having no missing values
- ✓ Results are explainable and defensible

---

## Congratulations!

You now understand:
- When and why to drop missing data
- When and why to fill missing data
- How to choose between strategies
- Common mistakes to avoid
- How to validate your cleaning

**You're ready to handle missing values responsibly in real datasets!**